In [1]:
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
import pickle
from sklearn.model_selection import train_test_split

In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,3,15619304,Onio,502,France,Female,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,4,15701354,Boni,699,France,Female,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43.0,2,125510.82,1,NaN,1.0,79084.10,0


In [3]:
# Preprocess the data
data=data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)


In [4]:
# Encode categorical variables
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,608,Spain,0,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,502,France,0,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,699,France,0,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,850,Spain,0,43.0,2,125510.82,1,NaN,1.0,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9997,709,France,0,36.0,7,0.00,1,0.0,1.0,42085.58,1
9998,772,Germany,1,42.0,3,75075.31,2,1.0,0.0,92888.52,1
9999,772,Germany,1,42.0,3,75075.31,2,1.0,0.0,92888.52,1
10000,792,France,0,28.0,4,130142.79,1,1.0,0.0,38190.78,0


In [5]:
data.isnull().sum()

CreditScore        0
Geography          1
Gender             0
Age                1
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          1
IsActiveMember     1
EstimatedSalary    0
Exited             0
dtype: int64

In [6]:
# data['HasCrCard'].value_counts()
#data['IsActiveMember'].value_counts()
data['Geography'].value_counts()

Geography
France     5014
Germany    2510
Spain      2477
Name: count, dtype: int64

In [7]:
data['Age']=data['Age'].fillna(data['Age'].mean())
data['HasCrCard']=data['HasCrCard'].fillna(1)
data['IsActiveMember']=data['IsActiveMember'].fillna(1)
data['Geography']=data['Geography'].fillna("France")


In [8]:
data.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [9]:
# OneHotEncode Geography 
onehot_encoder_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0
9999,0.0,1.0,0.0
10000,1.0,0.0,0.0


In [10]:
# Combine one_hot encoded columns with the original data
data=pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42.0,2,0.00,1,1.0,1.0,101348.88,1,1.0,0.0,0.0
1,608,0,41.0,1,83807.86,1,0.0,1.0,112542.58,0,0.0,0.0,1.0
2,502,0,42.0,8,159660.80,3,1.0,0.0,113931.57,1,1.0,0.0,0.0
3,699,0,39.0,1,0.00,2,0.0,0.0,93826.63,0,1.0,0.0,0.0
4,850,0,43.0,2,125510.82,1,1.0,1.0,79084.10,0,0.0,0.0,1.0


In [11]:
# Split the data into features and target
X=data.drop('EstimatedSalary', axis=1)
y=data['EstimatedSalary']


In [12]:
# Split the data in training and testing sets
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Scale these features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [14]:
# Save the encoders and scaler for later use
with open('label_encoder_gender.pk1', 'wb')as file:
    pickle.dump(label_encoder_gender, file)
    
with open('onehot_encoder_geo.pk1', 'wb')as file:
    pickle.dump(onehot_encoder_geo, file)
    
with open('scaler.pk1', 'wb')as file:
    pickle.dump(scaler, file)

In [15]:
# ANN IMPLEMENTATION

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [17]:
# Build the model
model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

# compile the model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

model.summary()

f:\QSP\Gen AI\deeplearning\DLENV\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Set up TensorBoard
log_dir="regressionlogs/fit/" +datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [19]:
##set up Early stopping
early_stopping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)


In [20]:
##Train the model 
history=model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=100,
    callbacks=[early_stopping_callback,tensorboard_callback]
)

Epoch 1/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 100529.0391 - mae: 100529.0391 - val_loss: 97966.1719 - val_mae: 97966.1719
Epoch 2/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 99960.5938 - mae: 99960.5938 - val_loss: 96817.3359 - val_mae: 96817.3359
Epoch 3/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 97954.1406 - mae: 97954.1406 - val_loss: 93880.2344 - val_mae: 93880.2344
Epoch 4/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 94004.0000 - mae: 94004.0000 - val_loss: 88937.9766 - val_mae: 88937.9766
Epoch 5/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 88006.8359 - mae: 88006.8359 - val_loss: 82185.4219 - val_mae: 82185.4219
Epoch 6/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - loss: 80481.7109 - mae: 80481.7109 - val_loss: 74447.2891 - val_mae: 74447.2891
Epoch 7/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 72461.0391 - mae: 72461.0391 - val_loss: 66789.5859 - val_mae: 66789.5859
Epoch 8/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 4s 8

In [21]:
%load_ext tensorboard

In [22]:
%tensorboard --logdir regressionlogs/fit

In [23]:
#Evaluate model on the test data
test_loss ,test_mae=model.evaluate(X_test,y_test)
print(f"Test mae :{test_mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 50294.5508 - mae: 50294.5508
Test mae :50294.55078125


In [24]:
model.save("regression_model.h5")